# Deep Portfolios with Autoencoders - exploration notebook

Interactive companion to `main.py`. It reuses the `src` package (no logic is
duplicated here). Run `python main.py` once first, or set `MIAX_AE_TOKEN`, so
the data cache in `data/raw/` exists.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from src.data.api_client import load_dataset
from src.pipeline import run_pipeline
from src.utils.config import DATA_DIR, RunConfig, get_api_url, get_token
from src.utils.logging_config import configure_logging

configure_logging()
dataset = load_dataset(get_api_url(), get_token(), DATA_DIR)
dataset.prices_wide.shape

## Run the corrected pipeline (denoising autoencoder, latent 8)

In [ ]:
result = run_pipeline(dataset, RunConfig())
result.comparison_frame()

In [ ]:
result.autoencoder_eval.validation_summary.to_frame("Autoencoder")

## Explainability: which company property does each latent axis encode?

In [ ]:
%matplotlib inline
from src.visualization import plots

selected = list(result.autoencoder_eval.weights)
plots.plot_latent_tsne_by_sector(result.latent, result.tickers, selected,
                                 dataset.sectors, 42, result.config.plots);

In [ ]:
plots.plot_latent_feature_gradients(result.latent, result.tickers, selected,
                                    result.features, result.config.plots);
plots.plot_latent_feature_heatmap(result.latent_feature_corr,
                                  result.config.plots);

## Latent-size sensitivity (use with care)

The latent size is a hyper-parameter chosen on validation. Look for the
"elbow" and stop there: tuning many variants against validation overfits to
validation, which the one-shot test evaluation then penalises.

In [ ]:
from dataclasses import replace

rows = {}
for dim in (2, 4, 8, 16):
    cfg = RunConfig(denoising=replace(RunConfig().denoising, latent_dim=dim))
    r = run_pipeline(dataset, cfg)
    rows[dim] = (r.autoencoder_eval.te_train, r.autoencoder_eval.te_validation)
rows